# Module 6: The Grand Capstone
## Project: Medical Research & Diagnosis Pipeline (LangGraph)

### **Goal**
We will build a high-complexity system where three specialized agents coordinate to solve a medical research request.
1. **The Researcher:** Searches external medical databases.
2. **The Analyst:** Interprets clinical findings from the "Memory Vault".
3. **The Consultant:** Acts as the Lead, reviewing all data and drafting a final professional report.

In [1]:
import os
from typing import TypedDict, List, Annotated
import operator

# 2026 Standard Orchestration
from langgraph.graph import StateGraph, END
from langchain_mistralai import ChatMistralAI
from langchain_core.messages import BaseMessage, HumanMessage

# Initialize Environment
os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY", "your-key-here")
llm = ChatMistralAI(model="mistral-large-latest", temperature=0)

c:\Users\zarya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\zarya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
c:\Users\zarya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Defining the State Machine

In [2]:
class AgentState(TypedDict):
    """The Shared Memory of the Pipeline"""
    messages: Annotated[List[BaseMessage], operator.add]
    research_data: str
    analysis_report: str
    final_summary: str
    audit_status: bool

### **Agent Persona Logic**
Each node in our graph represents a specialized persona. 
- The **Researcher** uses a tool to fetch information.
- The **Analyst** processes that info.
- The **Consultant** performs a final check. If the report is incomplete, the graph will **loop back** to the Researcher.

In [3]:
def researcher_node(state: AgentState):
    print("--- NODE: MEDICAL RESEARCHER ---")
    # Simulate searching PubMed or clinical trials
    return {"research_data": "Found new findings on Aortic Dissection treatment protocols."}

def analyst_node(state: AgentState):
    print("--- NODE: TECHNICAL ANALYST ---")
    return {"analysis_report": f"Processed findings: {state['research_data']}"}

def consultant_node(state: AgentState):
    print("--- NODE: SENIOR CONSULTANT (AUDITOR) ---")
    # Quality Audit Logic
    if "Aortic" not in state['research_data']:
        return {"audit_status": False, "final_summary": "Incomplete research."}
    return {"audit_status": True, "final_summary": "FINAL REPORT: Success."}

# Define Routing Logic
def should_continue(state: AgentState):
    if state["audit_status"]:
        return "end"
    return "refine"

# Construct the Orchestrator
workflow = StateGraph(AgentState)

workflow.add_node("researcher", researcher_node)
workflow.add_node("analyst", analyst_node)
workflow.add_node("consultant", consultant_node)

workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "analyst")
workflow.add_edge("analyst", "consultant")

workflow.add_conditional_edges(
    "consultant",
    should_continue,
    {"refine": "researcher", "end": END}
)

app = workflow.compile()

### **The Graduation Deliverable**
To complete the Capstone, you must:
1.  Wrap this `app` into the **FastAPI** `main.py` we built in Module 3.
2.  Deploy it using the **Docker Compose** stack from Module 5.
3.  Ensure the **Memory Vault** (Module 4) is connected to the Analyst node.

**Final Step:** Run the cell below to trigger the autonomous loop.

In [4]:
inputs = {"messages": [HumanMessage(content="Start Medical Audit")], "research_data": ""}

for output in app.stream(inputs):
    for key, value in output.items():
        print(f"Update from {key}: {value}")
    print("-" * 20)

--- NODE: MEDICAL RESEARCHER ---
Update from researcher: {'research_data': 'Found new findings on Aortic Dissection treatment protocols.'}
--------------------
--- NODE: TECHNICAL ANALYST ---
Update from analyst: {'analysis_report': 'Processed findings: Found new findings on Aortic Dissection treatment protocols.'}
--------------------
--- NODE: SENIOR CONSULTANT (AUDITOR) ---
Update from consultant: {'audit_status': True, 'final_summary': 'FINAL REPORT: Success.'}
--------------------
